# 4. Foundation Models

## Q4.1 Prompting an LLM to solve a time-series problem

### Q4.1.1 Transform each patient's time series data to a simple aggregated piece of text
We transform each patient's time series data to a simple aggregated piece of text. To
save valuable text and token space and run models faster, we came up with a
compact summary.

In [24]:
#Load the data
import numpy as np
import pandas as pd

df_train = pd.read_parquet("processed/set_a.parquet")
df_val   = pd.read_parquet("processed/set_b.parquet")
df_test  = pd.read_parquet("processed/set_c.parquet")

#note same function as in Exercise 2.1 to get the simple extracted features
DYNAMIC_VARS = sorted([
    'ALP', 'ALT', 'AST', 'Albumin', 'BUN', 'Bilirubin', 'Cholesterol',
    'Creatinine', 'DiasABP', 'FiO2', 'GCS', 'Glucose', 'HCO3', 'HCT',
    'HR', 'K', 'Lactate', 'MAP', 'MechVent', 'Mg', 'NIDiasABP', 'NIMAP',
    'NISysABP', 'Na', 'PaCO2', 'PaO2', 'Platelets', 'RespRate', 'SaO2',
    'SysABP', 'Temp', 'TroponinI', 'TroponinT', 'Urine', 'WBC', 'Weight', 'pH'
])
STATIC_VARS = ['Age', 'Gender', 'Height', 'StaticWeight']
ALL_FEATURES = DYNAMIC_VARS + STATIC_VARS  # 41 features total

def extract_simple_features(df):
    """
    Per patient:
      - Forward-fill measurements over time to ignore gaps/NaNs.
      - Dynamic variables: take the last non-null value after ffill.
      - Static variables: take as-is (they are constant).
    Returns one row per patient with features + label.
    """
    df_sorted = df.sort_values(["RecordID", "hour"])
    # Forward-fill within each patient, then grab the final row
    last_row = (
    df_sorted
    .groupby("RecordID")
    .apply(lambda g: g.ffill().iloc[-1])
    .reset_index()
)

    features = last_row[["RecordID"] + ALL_FEATURES].copy()
    labels   = last_row[["RecordID", "In-hospital_death"]].copy()
    return features, labels

X_train, y_train = extract_simple_features(df_train)
X_val,   y_val   = extract_simple_features(df_val)
X_test,  y_test  = extract_simple_features(df_test)


Creating a short summary of the data to be used for LLMs.

In [25]:
# Short-name to full-name and unit mappings for all available variables
FEATURE_FULL_NAMES = {
    "RecordID": "ICU stay identifier",
    "hour": "Hours since ICU admission",
    "ALP": "Alkaline phosphatase",
    "ALT": "Alanine transaminase",
    "AST": "Aspartate transaminase",
    "Albumin": "Serum albumin",
    "BUN": "Blood urea nitrogen",
    "Bilirubin": "Total bilirubin",
    "Cholesterol": "Total cholesterol",
    "Creatinine": "Serum creatinine",
    "DiasABP": "Invasive diastolic arterial blood pressure",
    "FiO2": "Fraction of inspired oxygen",
    "GCS": "Glasgow Coma Score",
    "Glucose": "Serum glucose",
    "HCO3": "Serum bicarbonate",
    "HCT": "Hematocrit",
    "HR": "Heart rate",
    "K": "Serum potassium",
    "Lactate": "Serum lactate",
    "MAP": "Invasive mean arterial blood pressure",
    "MechVent": "Mechanical ventilation flag",
    "Mg": "Serum magnesium",
    "NIDiasABP": "Non-invasive diastolic arterial blood pressure",
    "NIMAP": "Non-invasive mean arterial blood pressure",
    "NISysABP": "Non-invasive systolic arterial blood pressure",
    "Na": "Serum sodium",
    "PaCO2": "Partial pressure of arterial CO2",
    "PaO2": "Partial pressure of arterial O2",
    "Platelets": "Platelet count",
    "RespRate": "Respiration rate",
    "SaO2": "Oxygen saturation of hemoglobin",
    "SysABP": "Invasive systolic arterial blood pressure",
    "Temp": "Body temperature",
    "TroponinI": "Troponin I",
    "TroponinT": "Troponin T",
    "Urine": "Urine output",
    "WBC": "White blood cell count",
    "Weight": "Observed weight",
    "pH": "Arterial pH",
    "Age": "Age",
    "Gender": "Gender (0=female, 1=male)",
    "Height": "Height",
    "StaticWeight": "Admission weight (static)",
    "ICUType": "ICU type (1=CCU, 2=CSRU, 3=MICU, 4=SICU)",
    "In-hospital_death": "In-hospital death outcome"
}

FEATURE_UNITS = {
    "RecordID": "none",
    "hour": "hour",
    "ALP": "IU/L",
    "ALT": "IU/L",
    "AST": "IU/L",
    "Albumin": "g/dL",
    "BUN": "mg/dL",
    "Bilirubin": "mg/dL",
    "Cholesterol": "mg/dL",
    "Creatinine": "mg/dL",
    "DiasABP": "mmHg",
    "FiO2": "fraction (0-1)",
    "GCS": "score (3-15)",
    "Glucose": "mg/dL",
    "HCO3": "mmol/L",
    "HCT": "percent",
    "HR": "bpm",
    "K": "mEq/L",
    "Lactate": "mmol/L",
    "MAP": "mmHg",
    "MechVent": "binary (0/1)",
    "Mg": "mmol/L",
    "NIDiasABP": "mmHg",
    "NIMAP": "mmHg",
    "NISysABP": "mmHg",
    "Na": "mEq/L",
    "PaCO2": "mmHg",
    "PaO2": "mmHg",
    "Platelets": "cells/nL",
    "RespRate": "bpm",
    "SaO2": "percent",
    "SysABP": "mmHg",
    "Temp": "degC",
    "TroponinI": "ug/L",
    "TroponinT": "ug/L",
    "Urine": "mL",
    "WBC": "cells/nL",
    "Weight": "kg",
    "pH": "pH units",
    "Age": "years",
    "Gender": "binary (0/1)",
    "Height": "cm",
    "StaticWeight": "kg",
    "ICUType": "categorical (1-4)",
    "In-hospital_death": "binary (0/1)"
}

In [29]:
# craete the summary for the LLM to predict whether the patient is alive or not

def create_summary(row):
    summary = []
    for feature in ALL_FEATURES:
        value = row[feature]
        if pd.isna(value):
            continue  # Skip features with missing values
        summary.append(f"last known {FEATURE_FULL_NAMES[feature]} value: {value} {FEATURE_UNITS[feature]}.")
    return "; ".join(summary)

In [30]:
for _, row in X_train.iterrows():
    print(create_summary(row))
    break

last known Blood urea nitrogen value: 8.0 mg/dL.; last known Serum creatinine value: 0.7 mg/dL.; last known Glasgow Coma Score value: 15.0 score (3-15).; last known Serum glucose value: 115.0 mg/dL.; last known Serum bicarbonate value: 28.0 mmol/L.; last known Hematocrit value: 30.3 percent.; last known Heart rate value: 86.0 bpm.; last known Serum potassium value: 4.0 mEq/L.; last known Serum magnesium value: 1.9 mmol/L.; last known Non-invasive diastolic arterial blood pressure value: 55.0 mmHg.; last known Non-invasive mean arterial blood pressure value: 79.33 mmHg.; last known Non-invasive systolic arterial blood pressure value: 128.0 mmHg.; last known Serum sodium value: 136.0 mEq/L.; last known Platelet count value: 185.0 cells/nL.; last known Respiration rate value: 23.0 bpm.; last known Body temperature value: 37.8 degC.; last known Urine output value: 280.0 mL.; last known White blood cell count value: 9.4 cells/nL.; last known Observed weight value: -1.0 kg.; last known Age v

In [34]:
from ollama import chat

prompt = (
    "Based on the patient's last known measurements, predict in-hospital death.\n"
    "Output exactly one character: 0 if the patient is alive, 1 if deceased.\n"
    "Note the patient is in the ICU.\n"
    "No words, no punctuation, no explanation."
    f"\n\nSummary: {create_summary(X_train.iloc[0])}"
)

response = chat(
    model="gemma3:4b",
    messages=[{"role": "user", "content": prompt}],
    options={"temperature": 0, "top_p": 0.1},
)

text = response.message.content.strip()
if text not in {"0", "1"}:
    # fallback or parse the first digit
    text = "1" if "1" in text else "0"
print(text)

1


In [39]:
predictions = []
for num, row in X_train.iterrows():
    if num >= 100:  # Limit to first 100 patients for speed
        break
    prompt = (
    "Based on the patient's last known measurements, predict in-hospital death.\n"
    "Output exactly one character: 0 if the patient is alive, 1 if deceased.\n"
    "Note the patient is in the ICU. You previously predicted 100% mortality rate, even though the actually mortality rate was 15%. So use this to adjust your predictions.\n"
    "No words, no punctuation, no explanation."
    f"\n\nSummary: {create_summary(row)}"
    )

    response = chat(
        model="gemma3:4b",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0, "top_p": 0.1},
    )

    text = response.message.content.strip()
    if text not in {"0", "1"}:
        # fallback or parse the first digit
        text = "1" if "1" in text else "0"
    predictions.append(int(text))

average_prediction = np.mean(predictions)
print(f"Average predicted mortality rate: {average_prediction:.2f}")

Average predicted mortality rate: 0.46
